# DhakaRoadNet: Evaluation, Analysis, and Experiment Reporting

This notebook implements the evaluation-only stage for **DhakaRoadNet: An Edge AI System for Real-Time Road Object Detection Using a Custom Urban Traffic Dataset**.

It does not regenerate training code. It inspects trained YOLOv8 outputs, loads the best trained model when available, evaluates validation and test performance, analyzes learning curves and class weaknesses, generates predictions, and writes GitHub/CV-ready reports.

Main outputs:

- `reports/evaluation/evaluation_report.md`
- `reports/evaluation/results_summary.md`
- `reports/evaluation/metrics_table.csv`
- `reports/evaluation/class_metrics_table.csv`
- `reports/evaluation/failure_analysis.csv`
- `reports/evaluation/predictions/`
- `reports/evaluation/plots/`


## Current Workspace Finding

The project was inspected before this notebook was created. No trained `.pt` weights or training `results.csv` files were found under `model/` or `reports/`.

This notebook is still complete: it auto-discovers `best.pt` when it exists, produces clear missing-artifact warnings now, and runs full evaluation once trained weights are available. If training happened outside this workspace, set `MANUAL_BEST_MODEL_PATH` in the artifact discovery cell.


## Metric Concepts

- **True Positive (TP)**: predicted box matches a ground-truth object with correct class and enough IoU.
- **False Positive (FP)**: predicted object is wrong, duplicated, poorly localized, or background.
- **False Negative (FN)**: ground-truth object was missed.
- **Precision**: `TP / (TP + FP)`. High precision means fewer false alarms.
- **Recall**: `TP / (TP + FN)`. High recall means fewer missed objects.
- **F1 Score**: harmonic mean of precision and recall.
- **mAP50**: mean average precision at IoU 0.50.
- **mAP50-95**: stricter mean AP averaged over IoU thresholds 0.50 to 0.95.
- **Confusion matrix**: shows class confusion and background errors.
- **PR curve**: precision-recall tradeoff across confidence thresholds.


## 1. Imports and Project Paths


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import importlib.util
import json
import os
import platform
import random
import shutil
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_YAML = PROJECT_ROOT / "data" / "roboflow" / "data_yolov8.yaml"
MODEL_DIR = PROJECT_ROOT / "model"
RUNS_DIR = MODEL_DIR / "runs"
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"
REPORTS_DIR = PROJECT_ROOT / "reports"
EVAL_DIR = REPORTS_DIR / "evaluation"
EVAL_ARTIFACTS_DIR = EVAL_DIR / "artifacts"
EVAL_PLOTS_DIR = EVAL_DIR / "plots"
PREDICTIONS_DIR = EVAL_DIR / "predictions"
FAILURE_DIR = EVAL_DIR / "failure_cases"
BEST_DETECTIONS_DIR = EVAL_DIR / "best_detections"
CUSTOM_IMAGE_DIR = PROJECT_ROOT / "data" / "samples"
DEMO_DIR = PROJECT_ROOT / "demo"

for directory in [EVAL_DIR, EVAL_ARTIFACTS_DIR, EVAL_PLOTS_DIR, PREDICTIONS_DIR, FAILURE_DIR, BEST_DETECTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Dataset YAML       : {DATA_YAML}")
print(f"Runs directory     : {RUNS_DIR}")
print(f"Evaluation outputs : {EVAL_DIR}")
print(f"Python             : {sys.version.split()[0]}")
print(f"Platform           : {platform.platform()}")


## 2. Dependency Check


In [ ]:
def package_available(package_name: str) -> bool:
    return importlib.util.find_spec(package_name) is not None


dependency_status = pd.DataFrame(
    [
        {"package": "torch", "installed": package_available("torch")},
        {"package": "ultralytics", "installed": package_available("ultralytics")},
        {"package": "yaml", "installed": package_available("yaml")},
        {"package": "pandas", "installed": package_available("pandas")},
        {"package": "matplotlib", "installed": package_available("matplotlib")},
        {"package": "numpy", "installed": package_available("numpy")},
    ]
)
display(dependency_status)

if not package_available("torch") or not package_available("ultralytics"):
    print("Model evaluation requires torch and ultralytics. Reporting/template cells can still run.")


## 3. Load Dataset Configuration and Class Support


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_KEYS = {"train": "train", "val": "valid", "test": "test"}


def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing YAML file: {path}")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def normalize_names(names) -> list[str]:
    if isinstance(names, dict):
        return [names[k] for k in sorted(names)]
    return list(names)


def resolve_dataset_root(config: dict, yaml_path: Path) -> Path:
    root = Path(config.get("path", yaml_path.parent))
    if not root.is_absolute():
        root = (yaml_path.parent / root).resolve()
    return root


def resolve_split_image_dir(config: dict, yaml_path: Path, split_key: str) -> Path:
    dataset_root = resolve_dataset_root(config, yaml_path)
    split_path = Path(config[split_key])
    if split_path.is_absolute():
        return split_path
    return (dataset_root / split_path).resolve()


def list_images(image_dir: Path) -> list[Path]:
    if not image_dir.exists():
        return []
    return sorted(p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def scan_yolo_labels(config: dict, class_names: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    class_rows = []
    for yaml_key, split_name in SPLIT_KEYS.items():
        image_dir = resolve_split_image_dir(config, DATA_YAML, yaml_key)
        label_dir = image_dir.parent / "labels"
        images = list_images(image_dir)
        labels = sorted(label_dir.glob("*.txt")) if label_dir.exists() else []
        counts = Counter()
        areas = defaultdict(list)
        empty_labels = 0
        total_boxes = 0

        for label_path in labels:
            text = label_path.read_text(encoding="utf-8").strip()
            if not text:
                empty_labels += 1
                continue
            for line in text.splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                class_id = int(float(parts[0]))
                box_area = float(parts[3]) * float(parts[4])
                counts[class_id] += 1
                areas[class_id].append(box_area)
                total_boxes += 1

        summary_rows.append(
            {
                "split": split_name,
                "images": len(images),
                "labels": len(labels),
                "empty_labels": empty_labels,
                "boxes": total_boxes,
                "boxes_per_image": round(total_boxes / len(images), 3) if images else 0,
            }
        )
        for class_id, class_name in enumerate(class_names):
            class_areas = areas[class_id]
            class_rows.append(
                {
                    "split": split_name,
                    "class_id": class_id,
                    "class_name": class_name,
                    "instances": counts[class_id],
                    "median_box_area_norm": float(np.median(class_areas)) if class_areas else 0.0,
                    "mean_box_area_norm": float(np.mean(class_areas)) if class_areas else 0.0,
                }
            )
    return pd.DataFrame(summary_rows), pd.DataFrame(class_rows)


data_config = load_yaml(DATA_YAML)
class_names = normalize_names(data_config["names"])
if int(data_config["nc"]) != len(class_names):
    raise ValueError(f"nc={data_config['nc']} but names contains {len(class_names)} classes")

dataset_root = resolve_dataset_root(data_config, DATA_YAML)
split_rows = []
for yaml_key, split_name in SPLIT_KEYS.items():
    image_dir = resolve_split_image_dir(data_config, DATA_YAML, yaml_key)
    label_dir = image_dir.parent / "labels"
    split_rows.append(
        {
            "yaml_key": yaml_key,
            "split": split_name,
            "image_dir": str(image_dir),
            "label_dir": str(label_dir),
            "image_dir_exists": image_dir.exists(),
            "label_dir_exists": label_dir.exists(),
            "image_count": len(list_images(image_dir)),
            "label_count": len(list(label_dir.glob("*.txt"))) if label_dir.exists() else 0,
        }
    )

split_paths_df = pd.DataFrame(split_rows)
dataset_summary_df, dataset_class_stats_df = scan_yolo_labels(data_config, class_names)
class_support_df = (
    dataset_class_stats_df.pivot_table(index=["class_id", "class_name"], columns="split", values="instances", fill_value=0)
    .reset_index()
)
for split_name in ["train", "valid", "test"]:
    if split_name not in class_support_df.columns:
        class_support_df[split_name] = 0
class_support_df["total"] = class_support_df[["train", "valid", "test"]].sum(axis=1)
class_support_df = class_support_df.sort_values("total", ascending=True).reset_index(drop=True)

display(split_paths_df)
display(dataset_summary_df)
display(class_support_df.head(10))

split_paths_df.to_csv(EVAL_DIR / "dataset_split_paths.csv", index=False)
dataset_summary_df.to_csv(EVAL_DIR / "dataset_summary_for_evaluation.csv", index=False)
class_support_df.to_csv(EVAL_DIR / "class_support_table.csv", index=False)


## 4. Inspect Trained Model Outputs


In [ ]:
MANUAL_BEST_MODEL_PATH = None  # Example: r"C:\path\to\best.pt"
EXPERIMENT_NAME = "yolov8n_dhakaroadnet_baseline"


def discover_files(patterns: list[str], roots: list[Path]) -> list[Path]:
    found = []
    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            found.extend(root.rglob(pattern))
    return sorted(set(found), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)


def choose_best_model() -> Path | None:
    if MANUAL_BEST_MODEL_PATH:
        path = Path(MANUAL_BEST_MODEL_PATH)
        return path if path.exists() else None
    best_candidates = discover_files(["best.pt"], [RUNS_DIR, CHECKPOINT_DIR, MODEL_DIR])
    if best_candidates:
        return best_candidates[0]
    last_candidates = discover_files(["last.pt"], [RUNS_DIR, CHECKPOINT_DIR, MODEL_DIR])
    return last_candidates[0] if last_candidates else None


best_model_path = choose_best_model()
artifact_roots = [RUNS_DIR, CHECKPOINT_DIR, REPORTS_DIR]
inventory_rows = [
    {"artifact_type": "selected_model", "path": str(best_model_path) if best_model_path else None, "exists": bool(best_model_path)}
]
for pattern, artifact_type in [
    ("best.pt", "best_weight"),
    ("last.pt", "last_weight"),
    ("results.csv", "training_or_eval_results"),
    ("results.png", "results_plot"),
    ("confusion_matrix.png", "confusion_matrix"),
    ("PR_curve.png", "pr_curve"),
    ("P_curve.png", "precision_curve"),
    ("R_curve.png", "recall_curve"),
    ("F1_curve.png", "f1_curve"),
]:
    for path in discover_files([pattern], artifact_roots):
        inventory_rows.append({"artifact_type": artifact_type, "path": str(path), "exists": path.exists()})

artifact_inventory_df = pd.DataFrame(inventory_rows)
artifact_inventory_df.to_csv(EVAL_DIR / "artifact_inventory.csv", index=False)
display(artifact_inventory_df)

if best_model_path is None:
    print("No trained best.pt/last.pt found. Full evaluation is blocked until training outputs exist.")
else:
    print(f"Selected model for evaluation: {best_model_path}")


## 5. Device Selection


In [ ]:
def select_device() -> str | int:
    if not package_available("torch"):
        return "cpu"
    import torch
    return 0 if torch.cuda.is_available() else "cpu"


def device_description(device: str | int) -> str:
    if not package_available("torch"):
        return "CPU placeholder until PyTorch is installed"
    import torch
    if device == "cpu":
        return "CPU"
    return torch.cuda.get_device_name(device)


DEVICE = select_device()
print(f"Evaluation device: {DEVICE}")
print(f"Device name      : {device_description(DEVICE)}")


## 6. Load Best Trained Model


In [ ]:
LOAD_MODEL = False  # Set True after weights and ultralytics are available.

model = None
if LOAD_MODEL:
    if best_model_path is None or not best_model_path.exists():
        raise FileNotFoundError("No best model found. Train the model or set MANUAL_BEST_MODEL_PATH.")
    if not package_available("ultralytics"):
        raise ImportError("Install ultralytics before loading the model.")
    from ultralytics import YOLO
    model = YOLO(str(best_model_path))
    print(f"Loaded model: {best_model_path}")
else:
    print("LOAD_MODEL is False. Set it to True when trained weights are available.")


## 7. Evaluate Validation and Test Sets


In [ ]:
RUN_EVALUATION = False  # Set True after LOAD_MODEL can succeed.

EVAL_CONFIG = {
    "data": str(DATA_YAML),
    "imgsz": 640,
    "batch": 16,
    "conf": 0.001,
    "iou": 0.70,
    "device": DEVICE,
    "plots": True,
    "save_json": False,
    "project": str(EVAL_ARTIFACTS_DIR),
}


def safe_float(value):
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None


def extract_ultralytics_metrics(metrics, split_name: str, class_names: list[str]) -> tuple[dict, pd.DataFrame]:
    box = getattr(metrics, "box", None)
    precision = safe_float(getattr(box, "mp", None))
    recall = safe_float(getattr(box, "mr", None))
    f1 = 2 * precision * recall / (precision + recall) if precision is not None and recall is not None and (precision + recall) > 0 else None
    scalar_row = {
        "split": split_name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "map50": safe_float(getattr(box, "map50", None)),
        "map50_95": safe_float(getattr(box, "map", None)),
        "fitness": safe_float(getattr(metrics, "fitness", None)),
        "save_dir": str(getattr(metrics, "save_dir", "")),
    }

    maps = getattr(box, "maps", None)
    p_values = getattr(box, "p", None)
    r_values = getattr(box, "r", None)
    class_rows = []
    for class_id, class_name in enumerate(class_names):
        row = {"split": split_name, "class_id": class_id, "class_name": class_name}
        row["map50_95"] = safe_float(maps[class_id]) if maps is not None and len(maps) > class_id else None
        row["precision"] = safe_float(p_values[class_id]) if p_values is not None and len(p_values) > class_id else None
        row["recall"] = safe_float(r_values[class_id]) if r_values is not None and len(r_values) > class_id else None
        row["f1"] = 2 * row["precision"] * row["recall"] / (row["precision"] + row["recall"]) if row["precision"] is not None and row["recall"] is not None and (row["precision"] + row["recall"]) > 0 else None
        class_rows.append(row)
    return scalar_row, pd.DataFrame(class_rows)


evaluation_summary_df = pd.DataFrame()
evaluation_class_metrics_df = pd.DataFrame()

if RUN_EVALUATION:
    if model is None:
        if best_model_path is None or not best_model_path.exists():
            raise FileNotFoundError("No trained model found. Train first or set MANUAL_BEST_MODEL_PATH.")
        if not package_available("ultralytics"):
            raise ImportError("Install ultralytics before evaluation.")
        from ultralytics import YOLO
        model = YOLO(str(best_model_path))

    scalar_rows = []
    class_frames = []
    for split_key, split_label in [("val", "validation"), ("test", "test")]:
        metrics = model.val(**EVAL_CONFIG, split=split_key, name=f"{EXPERIMENT_NAME}_{split_label}")
        scalar_row, class_df = extract_ultralytics_metrics(metrics, split_label, class_names)
        scalar_rows.append(scalar_row)
        class_frames.append(class_df)

    evaluation_summary_df = pd.DataFrame(scalar_rows)
    evaluation_class_metrics_df = pd.concat(class_frames, ignore_index=True)
    evaluation_summary_df.to_csv(EVAL_DIR / "metrics_table.csv", index=False)
    evaluation_class_metrics_df.to_csv(EVAL_DIR / "class_metrics_table.csv", index=False)
    display(evaluation_summary_df)
    display(evaluation_class_metrics_df.head())
else:
    print("RUN_EVALUATION is False. Evaluation pipeline is ready but was not executed.")


## 8. Collect Confusion Matrix and Curve Artifacts


In [ ]:
COLLECT_EVALUATION_ARTIFACTS = False
PLOT_PATTERNS = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png",
    "F1_curve.png",
]


def copy_evaluation_plots(source_roots: list[Path], destination: Path) -> pd.DataFrame:
    copied_rows = []
    destination.mkdir(parents=True, exist_ok=True)
    for source_root in source_roots:
        if not source_root.exists():
            continue
        for pattern in PLOT_PATTERNS:
            for src in source_root.rglob(pattern):
                dst = destination / f"{src.parent.name}_{src.name}"
                shutil.copy2(src, dst)
                copied_rows.append({"source": str(src), "copied_to": str(dst), "artifact": src.name})
    return pd.DataFrame(copied_rows)


if COLLECT_EVALUATION_ARTIFACTS:
    copied_artifacts_df = copy_evaluation_plots([EVAL_ARTIFACTS_DIR, RUNS_DIR], EVAL_PLOTS_DIR)
    copied_artifacts_df.to_csv(EVAL_DIR / "copied_evaluation_artifacts.csv", index=False)
    display(copied_artifacts_df)
else:
    print("COLLECT_EVALUATION_ARTIFACTS is False. Enable it after RUN_EVALUATION completes.")


## 9. Learning Curve Analysis: Overfitting vs Underfitting


In [ ]:
def find_training_results_csv() -> Path | None:
    candidates = discover_files(["results.csv"], [RUNS_DIR, REPORTS_DIR])
    return candidates[0] if candidates else None


def summarize_learning_curves(results_csv: Path) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(results_csv)
    df.columns = [col.strip() for col in df.columns]
    summary = {"results_csv": str(results_csv), "epochs_recorded": int(len(df))}
    for col in df.columns:
        if "loss" not in col.lower() and "map" not in col.lower() and "precision" not in col.lower() and "recall" not in col.lower():
            continue
        values = pd.to_numeric(df[col], errors="coerce").dropna()
        if values.empty:
            continue
        summary[f"{col}_first"] = float(values.iloc[0])
        summary[f"{col}_last"] = float(values.iloc[-1])
        summary[f"{col}_best"] = float(values.min() if "loss" in col.lower() else values.max())
    return df, summary


training_results_csv = find_training_results_csv()
learning_curve_summary = {"status": "missing_training_results_csv"}
training_results_df = pd.DataFrame()

if training_results_csv:
    training_results_df, learning_curve_summary = summarize_learning_curves(training_results_csv)
    display(training_results_df.tail())
    with (EVAL_DIR / "learning_curve_summary.json").open("w", encoding="utf-8") as f:
        json.dump(learning_curve_summary, f, indent=2)
    loss_cols = [col for col in training_results_df.columns if "loss" in col.lower()]
    if loss_cols and "epoch" in training_results_df.columns:
        ax = training_results_df.plot(x="epoch", y=loss_cols, figsize=(12, 5), title="Training vs Validation Loss")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        path = EVAL_PLOTS_DIR / "training_validation_loss_curves.png"
        plt.savefig(path, dpi=180)
        plt.show()
        print(f"Saved loss curve plot: {path}")
else:
    print("No training results.csv found. Learning-curve analysis will remain pending.")


## 10. Prediction Generation: Validation, Test, Custom Images, Videos


In [ ]:
RUN_PREDICTIONS = False
PREDICTION_CONF = 0.25
PREDICTION_IOU = 0.70
MAX_SAMPLE_IMAGES_PER_SPLIT = 24


def sample_split_images(split_key: str, max_count: int = MAX_SAMPLE_IMAGES_PER_SPLIT) -> list[Path]:
    image_dir = resolve_split_image_dir(data_config, DATA_YAML, split_key)
    images = list_images(image_dir)
    if not images:
        return []
    rng = random.Random(SEED)
    return rng.sample(images, min(max_count, len(images)))


def discover_custom_sources() -> dict[str, list[Path]]:
    images = list_images(CUSTOM_IMAGE_DIR) if CUSTOM_IMAGE_DIR.exists() else []
    screenshots = list_images(DEMO_DIR / "screenshots") if (DEMO_DIR / "screenshots").exists() else []
    videos = sorted(DEMO_DIR.glob("*.mp4")) if DEMO_DIR.exists() else []
    return {"custom_images": images + screenshots, "videos": videos}


prediction_sources = {
    "validation_images": sample_split_images("val"),
    "test_images": sample_split_images("test"),
    **discover_custom_sources(),
}
prediction_sources_df = pd.DataFrame(
    [{"source_group": group, "path": str(path)} for group, paths in prediction_sources.items() for path in paths]
)
prediction_sources_df.to_csv(EVAL_DIR / "prediction_sources.csv", index=False)
display(prediction_sources_df.groupby("source_group").size().reset_index(name="count") if not prediction_sources_df.empty else prediction_sources_df)

if RUN_PREDICTIONS:
    if model is None:
        if best_model_path is None or not best_model_path.exists():
            raise FileNotFoundError("No trained model found. Train first or set MANUAL_BEST_MODEL_PATH.")
        if not package_available("ultralytics"):
            raise ImportError("Install ultralytics before predictions.")
        from ultralytics import YOLO
        model = YOLO(str(best_model_path))

    rows = []
    for group, paths in prediction_sources.items():
        if not paths:
            continue
        run_name = f"{EXPERIMENT_NAME}_{group}"
        model.predict(
            source=[str(path) for path in paths],
            conf=PREDICTION_CONF,
            iou=PREDICTION_IOU,
            imgsz=640,
            device=DEVICE,
            save=True,
            save_txt=True,
            save_conf=True,
            project=str(PREDICTIONS_DIR),
            name=run_name,
            exist_ok=True,
        )
        rows.append({"source_group": group, "run_dir": str(PREDICTIONS_DIR / run_name), "items": len(paths)})
    prediction_runs_df = pd.DataFrame(rows)
    prediction_runs_df.to_csv(EVAL_DIR / "prediction_runs.csv", index=False)
    display(prediction_runs_df)
else:
    print("RUN_PREDICTIONS is False. Prediction pipeline is ready but was not executed.")


## 11. Detection-Level TP/FP/FN Error Analysis


In [ ]:
RUN_ERROR_ANALYSIS = False
ERROR_SPLIT_KEY = "test"
ERROR_CONF_THRESHOLD = 0.25
ERROR_IOU_THRESHOLD = 0.50
MAX_ERROR_IMAGES = 200


def xywhn_to_xyxy(box: list[float], width: int, height: int) -> list[float]:
    x, y, w, h = box
    return [(x - w / 2) * width, (y - h / 2) * height, (x + w / 2) * width, (y + h / 2) * height]


def box_iou(a: list[float], b: list[float]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def load_ground_truth_records(image_path: Path, class_names: list[str]) -> list[dict]:
    import cv2
    image = cv2.imread(str(image_path))
    if image is None:
        return []
    height, width = image.shape[:2]
    label_path = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
    records = []
    if not label_path.exists():
        return records
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        class_id = int(float(parts[0]))
        records.append({"class_id": class_id, "class_name": class_names[class_id], "box": xywhn_to_xyxy(list(map(float, parts[1:5])), width, height)})
    return records


def result_to_prediction_records(result, class_names: list[str]) -> list[dict]:
    if result.boxes is None:
        return []
    boxes = result.boxes.xyxy.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy()
    confs = result.boxes.conf.cpu().numpy()
    rows = []
    for box, cls, conf in zip(boxes, classes, confs):
        class_id = int(cls)
        rows.append({"class_id": class_id, "class_name": class_names[class_id], "confidence": float(conf), "box": list(map(float, box))})
    return sorted(rows, key=lambda row: row["confidence"], reverse=True)


def match_predictions(predictions: list[dict], ground_truths: list[dict], threshold: float):
    matched_gt = set()
    tp, fp = [], []
    for pred in predictions:
        best_iou, best_idx = 0.0, None
        for idx, gt in enumerate(ground_truths):
            if idx in matched_gt or gt["class_id"] != pred["class_id"]:
                continue
            iou = box_iou(pred["box"], gt["box"])
            if iou > best_iou:
                best_iou, best_idx = iou, idx
        if best_idx is not None and best_iou >= threshold:
            matched_gt.add(best_idx)
            tp.append({**pred, "iou": best_iou})
        else:
            fp.append({**pred, "best_iou": best_iou})
    fn = [gt for idx, gt in enumerate(ground_truths) if idx not in matched_gt]
    return tp, fp, fn


failure_analysis_df = pd.DataFrame()
worst_classes_df = pd.DataFrame()

if RUN_ERROR_ANALYSIS:
    if model is None:
        if best_model_path is None or not best_model_path.exists():
            raise FileNotFoundError("No trained model found. Train first or set MANUAL_BEST_MODEL_PATH.")
        if not package_available("ultralytics"):
            raise ImportError("Install ultralytics before error analysis.")
        from ultralytics import YOLO
        model = YOLO(str(best_model_path))

    image_paths = sample_split_images(ERROR_SPLIT_KEY, MAX_ERROR_IMAGES)
    class_counts = {class_id: {"tp": 0, "fp": 0, "fn": 0} for class_id in range(len(class_names))}
    failure_rows = []
    for result in model.predict(source=[str(path) for path in image_paths], conf=ERROR_CONF_THRESHOLD, iou=PREDICTION_IOU, imgsz=640, device=DEVICE, verbose=False):
        image_path = Path(result.path)
        gt = load_ground_truth_records(image_path, class_names)
        pred = result_to_prediction_records(result, class_names)
        tps, fps, fns = match_predictions(pred, gt, ERROR_IOU_THRESHOLD)
        for row in tps:
            class_counts[row["class_id"]]["tp"] += 1
        for row in fps:
            class_counts[row["class_id"]]["fp"] += 1
            failure_rows.append({"image": str(image_path), "error_type": "false_positive", "class_id": row["class_id"], "class_name": row["class_name"], "confidence": row["confidence"], "iou": row["best_iou"]})
        for row in fns:
            class_counts[row["class_id"]]["fn"] += 1
            failure_rows.append({"image": str(image_path), "error_type": "false_negative", "class_id": row["class_id"], "class_name": row["class_name"], "confidence": None, "iou": None})

    metric_rows = []
    for class_id, class_name in enumerate(class_names):
        tp, fp, fn = class_counts[class_id]["tp"], class_counts[class_id]["fp"], class_counts[class_id]["fn"]
        precision = tp / (tp + fp) if (tp + fp) else None
        recall = tp / (tp + fn) if (tp + fn) else None
        f1 = 2 * precision * recall / (precision + recall) if precision is not None and recall is not None and (precision + recall) else None
        metric_rows.append({"class_id": class_id, "class_name": class_name, "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1})

    failure_analysis_df = pd.DataFrame(failure_rows)
    worst_classes_df = pd.DataFrame(metric_rows).sort_values(["f1", "fn", "fp"], ascending=[True, False, False])
    failure_analysis_df.to_csv(EVAL_DIR / "failure_analysis.csv", index=False)
    worst_classes_df.to_csv(EVAL_DIR / "worst_classes.csv", index=False)
    display(worst_classes_df.head(10))
else:
    print("RUN_ERROR_ANALYSIS is False. TP/FP/FN analysis is ready but was not executed.")


## 12. Weak-Class Diagnosis


In [ ]:
def diagnose_class_weakness(class_id: int, class_name: str, support_row: pd.Series | None) -> str:
    reasons = []
    train_count = int(support_row.get("train", 0)) if support_row is not None else 0
    total_count = int(support_row.get("total", 0)) if support_row is not None else 0
    area_rows = dataset_class_stats_df[(dataset_class_stats_df["class_id"] == class_id) & (dataset_class_stats_df["split"] == "train")]
    median_area = float(area_rows["median_box_area_norm"].iloc[0]) if not area_rows.empty else 0.0
    lower_name = class_name.lower()

    if total_count < 50:
        reasons.append("very limited total samples")
    elif train_count < 100:
        reasons.append("limited training samples")
    if 0 < median_area < 0.002:
        reasons.append("small object size")
    if lower_name in {"human", "motorbike", "rickshaw", "bus", "car"}:
        reasons.append("likely occlusion and crowding")
    if lower_name in {"pothole", "manhole", "speed breaker", "zebra crossing"}:
        reasons.append("road-surface appearance varies with lighting and camera angle")
    if lower_name in {"police car", "pickup truck", "leguna", "mini truck"}:
        reasons.append("rare class and visual similarity with other vehicles")
    if not reasons:
        reasons.append("inspect confusion matrix and examples")
    return "; ".join(reasons)


support_lookup = {int(row["class_id"]): row for _, row in class_support_df.iterrows()}
weakness_rows = []
for class_id, class_name in enumerate(class_names):
    support_row = support_lookup.get(class_id)
    weakness_rows.append(
        {
            "class_id": class_id,
            "class_name": class_name,
            "train_instances": int(support_row.get("train", 0)) if support_row is not None else 0,
            "valid_instances": int(support_row.get("valid", 0)) if support_row is not None else 0,
            "test_instances": int(support_row.get("test", 0)) if support_row is not None else 0,
            "total_instances": int(support_row.get("total", 0)) if support_row is not None else 0,
            "likely_weakness_reasons": diagnose_class_weakness(class_id, class_name, support_row),
        }
    )

weakness_diagnosis_df = pd.DataFrame(weakness_rows).sort_values(["total_instances", "class_id"]).reset_index(drop=True)
weakness_diagnosis_df.to_csv(EVAL_DIR / "weak_class_diagnosis.csv", index=False)
display(weakness_diagnosis_df.head(12))


## 13. Generate Research-Style Reports


In [ ]:
GENERATE_REPORTS = True


def markdown_table(df: pd.DataFrame, max_rows: int = 20) -> str:
    if df is None or df.empty:
        return "_Pending: run evaluation to populate this table._"
    view = df.head(max_rows).copy()
    view = view.fillna("Pending")
    columns = [str(col) for col in view.columns]
    rows = ["| " + " | ".join(columns) + " |"]
    rows.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for _, row in view.iterrows():
        values = [str(row[col]).replace("|", "/") for col in view.columns]
        rows.append("| " + " | ".join(values) + " |")
    return "\n".join(rows)


def metric_value(df: pd.DataFrame, split: str, column: str) -> str:
    if df.empty or column not in df.columns:
        return "Pending"
    rows = df[df["split"] == split]
    if rows.empty:
        return "Pending"
    value = rows.iloc[0][column]
    return "Pending" if pd.isna(value) else f"{float(value):.4f}"


def existing_plot_reference(name_part: str) -> str:
    candidates = sorted(EVAL_PLOTS_DIR.glob(f"*{name_part}*"))
    if not candidates:
        return "Pending: run evaluation with plots enabled."
    return str(candidates[0].relative_to(PROJECT_ROOT)).replace("\\", "/")


def write_reports():
    now = datetime.now().isoformat(timespec="seconds")
    model_text = str(best_model_path) if best_model_path else "Pending: no best.pt discovered"
    status = learning_curve_summary.get("status", "available") if isinstance(learning_curve_summary, dict) else "available"

    evaluation_report = f'''
# DhakaRoadNet Evaluation Report

Generated: {now}

## Evaluation Objective

This report evaluates the YOLOv8 road-object detector for DhakaRoadNet as a mini research project. The goal is to measure detection quality, diagnose model weaknesses, and decide whether the trained model is ready for Android/TFLite deployment.

## Model and Dataset

- Model weights: `{model_text}`
- Dataset YAML: `{DATA_YAML}`
- Number of classes: {len(class_names)}
- Dataset root: `{dataset_root}`

### Dataset Split Summary

{markdown_table(dataset_summary_df)}

### Lowest-Support Classes

{markdown_table(class_support_df.head(10))}

## Core Metrics

| Split | Precision | Recall | F1 | mAP50 | mAP50-95 |
|---|---:|---:|---:|---:|---:|
| Validation | {metric_value(evaluation_summary_df, 'validation', 'precision')} | {metric_value(evaluation_summary_df, 'validation', 'recall')} | {metric_value(evaluation_summary_df, 'validation', 'f1')} | {metric_value(evaluation_summary_df, 'validation', 'map50')} | {metric_value(evaluation_summary_df, 'validation', 'map50_95')} |
| Test | {metric_value(evaluation_summary_df, 'test', 'precision')} | {metric_value(evaluation_summary_df, 'test', 'recall')} | {metric_value(evaluation_summary_df, 'test', 'f1')} | {metric_value(evaluation_summary_df, 'test', 'map50')} | {metric_value(evaluation_summary_df, 'test', 'map50_95')} |

## Diagnostic Plots

- Confusion matrix: `{existing_plot_reference('confusion_matrix')}`
- PR curve: `{existing_plot_reference('PR_curve')}`
- Precision curve: `{existing_plot_reference('P_curve')}`
- Recall curve: `{existing_plot_reference('R_curve')}`
- F1 curve: `{existing_plot_reference('F1_curve')}`

## Weak-Class Diagnosis

{markdown_table(weakness_diagnosis_df.head(12))}

Likely causes to inspect:

- Class imbalance, especially rare classes.
- Small object size for distant pedestrians, vehicles, and road-surface defects.
- Occlusion and crowding in dense Dhaka traffic scenes.
- Annotation ambiguity between visually similar vehicles.
- Lighting, road texture, and camera-motion variation for potholes, manholes, speed breakers, and zebra crossings.

## Learning Curve Interpretation

Training results status: `{status}`

- Overfitting: train loss decreases while validation loss rises or validation mAP falls.
- Underfitting: both losses remain high and mAP remains low.
- Healthy training: train and validation losses decrease together and mAP improves before plateauing.

## False Positives, False Negatives, and True Positives

- True positives indicate correctly localized and classified road objects.
- False positives may come from background clutter, similar vehicle classes, duplicate boxes, shadows, or reflections.
- False negatives may come from small objects, occlusion, motion blur, low contrast, rare classes, or missing annotation patterns.

Detection-level failure file: `{(EVAL_DIR / 'failure_analysis.csv').relative_to(PROJECT_ROOT) if (EVAL_DIR / 'failure_analysis.csv').exists() else 'Pending: run error analysis.'}`

## Prediction Gallery

Prediction outputs are saved under `{PREDICTIONS_DIR.relative_to(PROJECT_ROOT)}` after `RUN_PREDICTIONS=True`.

Recommended galleries:

- validation predictions
- test predictions
- custom images
- sample videos
- best detections
- failure cases

## Key Findings

Pending until model evaluation is executed.

## Limitations

- Final numerical conclusions require trained weights and completed validation/test evaluation.
- Rare classes may not have enough validation/test examples for stable class-wise conclusions.
- mAP does not directly measure Android latency, memory, or battery impact.
- Offline images may not fully represent live camera motion blur and compression.

## Future Work

- Add more samples for rare and confused classes.
- Improve annotation consistency for visually similar vehicle categories.
- Evaluate YOLOv8s if YOLOv8n underfits.
- Tune confidence threshold based on precision-recall tradeoff.
- Export to TFLite and benchmark on target Android hardware.

## Android/TFLite Readiness Checklist

- [ ] `best.pt` exists and is reproducible from the training notebook.
- [ ] Validation and test mAP50/mAP50-95 are recorded.
- [ ] Precision/recall tradeoff is acceptable for the app use case.
- [ ] Worst classes are understood and documented.
- [ ] Confusion matrix and PR/F1/P/R curves are saved.
- [ ] Prediction gallery contains successful and failed examples.
- [ ] Model is exported to TFLite.
- [ ] INT8 quantization is tested if latency/model size matters.
- [ ] Android inference latency is measured on target hardware.
- [ ] Labels file matches the training class order exactly.
- [ ] Camera preprocessing matches YOLO training image resizing expectations.
'''.strip() + "\n"

    results_summary = f'''
# DhakaRoadNet Results Summary

## Snapshot

- Model: `{model_text}`
- Dataset: `{DATA_YAML}`
- Classes: {len(class_names)}
- Validation mAP50: {metric_value(evaluation_summary_df, 'validation', 'map50')}
- Validation mAP50-95: {metric_value(evaluation_summary_df, 'validation', 'map50_95')}
- Test mAP50: {metric_value(evaluation_summary_df, 'test', 'map50')}
- Test mAP50-95: {metric_value(evaluation_summary_df, 'test', 'map50_95')}

## Metrics Table

{markdown_table(evaluation_summary_df)}

## Class Weakness Summary

{markdown_table(weakness_diagnosis_df.head(10))}

## GitHub Figure References

- Class distribution: `reports/figures/class_distribution.png`
- Confusion matrix: `{existing_plot_reference('confusion_matrix')}`
- PR curve: `{existing_plot_reference('PR_curve')}`
- Sample predictions: `{PREDICTIONS_DIR.relative_to(PROJECT_ROOT)}`

## Key Findings

Pending until model evaluation is executed.

## Limitations

Pending final quantitative and qualitative review.

## Future Work

- Evaluate YOLOv8s after YOLOv8n baseline.
- Add rare-class data.
- Export and benchmark TFLite model on Android.
'''.strip() + "\n"

    report_path = EVAL_DIR / "evaluation_report.md"
    summary_path = EVAL_DIR / "results_summary.md"
    report_path.write_text(evaluation_report, encoding="utf-8")
    summary_path.write_text(results_summary, encoding="utf-8")
    return report_path, summary_path


if GENERATE_REPORTS:
    report_path, summary_path = write_reports()
    print(f"Wrote evaluation report: {report_path}")
    print(f"Wrote results summary  : {summary_path}")


## Final Research Checklist

Before presenting this project on GitHub, CV, or higher-study applications, confirm:

- Evaluation uses a held-out test set, not only validation.
- Test metrics are reported separately from validation metrics.
- Confusion matrix and class-wise metrics are included.
- Worst classes are explained using evidence, not guesses.
- Prediction gallery includes both success and failure cases.
- Limitations are honest and specific.
- Android readiness is based on both accuracy and device latency.
